# 03 - Intervention parsing

* The goal of this notebook is to split each raw normalized .txt file into individual interventions (and their authors).

* Thanks to the strict formatting conventions used in the transcripts, a regular expression (regex) can be used to efficiently parse each intervention.


> **INPUT:** `data/raw/normalized_text/PL_<leg>_<num>.txt` — files, each containing the normalized text of a single session  

> **OUTPUT:** `data/raw/interventions/PL_<leg>_<num>.txt` — files, one per session, with content split by interventions

* *Due to the large number of files, only a few samples are included in the repository. To fully replicate the process, update the source/destination paths in the notebook's first cell.*



In [ ]:
from pathlib import Path

# Change the raw normalized txt files path HERE:
BASE_TEXT_DIR = Path('..') / 'data' / 'raw' / 'normalized_texts'

# Change the output path for the interventions HERE:   
OUT_INTERV_DIR = Path('..') / 'data' / 'raw' / 'interventions'

OUT_INTERV_DIR.mkdir(parents=True, exist_ok=True)

# Chosen regex to find each intervention
SPEAKER_REGEX = r'El señor [A-ZÑÁÉÍÓÚÜ][A-ZÑÁÉÍÓÚÜ]+|La señora [A-ZÑÁÉÍÓÚÜ][A-ZÑÁÉÍÓÚÜ]+'


In [2]:
# Reads a raw txt file and return a string

def get_text(path: Path) -> str:
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        print('File not found:', path)
        return ''
    except Exception as e:
        print('Unexpected error reading', path, e)
        return ''

In [74]:
import re
from typing import List, Tuple

def get_interventions(text: str, regex: str = SPEAKER_REGEX) -> List[Tuple[str, str]]:
    # Extract a list of (speaker_raw, intervention_text) tuples from a full text.
    out = []
    if not text:
        return []
    
    results = []
    

    pattern = r'((?:El señor|La señora)\s+[A-ZÑÁÉÍÓÚÜ][A-ZÑÁÉÍÓÚÜ\-\s\(\)\.]+?):'
    
    matches = list(re.finditer(pattern, text))
    
    if not matches:
        return []
    
    # Process each speaker-intervention pair
    for i in range(len(matches)):
        match = matches[i]
        
        # Extract speaker name (everything after "El señor/La señora" until ':')
        speaker_text = match.group(1).strip()
        
        # Skip if speaker text is empty or too short
        if not speaker_text or len(speaker_text) < 2:
            continue
        
        speaker = f"{match.group(0)[:-1]}:"  # "El señor GARCÍA:" (without the intervention)
        speaker = text[match.start():match.end()-1].strip()
        
        # Intervention text starts right after the colon
        intervention_start = match.end()
        
        # Intervention ends at the next speaker marker, or end of text
        if i + 1 < len(matches):
            intervention_end = matches[i + 1].start()
        else:
            intervention_end = len(text)
        
        intervention = text[intervention_start:intervention_end].strip()
        if intervention:
            results.append((speaker, intervention))
    
    return results


In [75]:
# You might choose a sample text to test the regex and function before running on all files
SAMPLE_TEXT = BASE_TEXT_DIR / 'PL_1_10.txt'

if SAMPLE_TEXT.exists():
    txt = get_text(SAMPLE_TEXT)
    interventions = get_interventions(txt)
    print('Found', len(interventions), 'interventions in', SAMPLE_TEXT.name)
    print(interventions[0])
    print(interventions[1])
    print(interventions[2])
    print(interventions[3])
    print(interventions[4])
    print(interventions[5])
    print(interventions[6])
else:
    print('Sample text not found; update SAMPLE_TEXT parameter to test locally')

Found 47 interventions in PL_1_10.txt
('El señor PRESIDENTE', 'Como saben Sus Señorías, en el orden del día de esta sesión, anunciado en la plenaria de la pasada semana figuraba incluido, con carácter provisional, un debate político sobre el tema de seguridad ciudadana, una proposición no de ley presentada por el Grupo Parlamentario Comunista sobre la reforma sanitaria ; una pregunta formulada por el Diputado señor Barón Crespo y otros dos señores Diputados acerca del incumplimiento de medidas de seguridad en aeropuertos y, finalmente, el señalamiento para la celebración de la siguiente sesión ordinaria. La Junta de Portavoces, en su reunión del día de ayer, acordó la introducción, además, de dos proposiciones de ley, una del Grupo Parlamentario Comunista y otra del Grupo Parlamentario Socialistas del Congreso, que se refieren ambas a la derogación del Real Decreto-ley 3-1979, de 26 de enero, sobre protección de la seguridad ciudadana. Hubo acuerdo de los Grupos Parlamentarios, hubo co

In [76]:
# Batch: recorrer carpeta de textos y generar archivos de intervenciones por sesión
def batch_interventions_to_files(src_dir: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    processed = 0
    for p in sorted(src_dir.glob('PL_*.txt')):
        txt = get_text(p)
        interventions = get_interventions(txt)
        if not interventions:
            continue
        out_path = out_dir / p.name
        with open(out_path, 'w', encoding='utf-8') as fh:
            for (speaker, text_block) in interventions:
                fh.write(speaker.strip())
                fh.write('\n')
                fh.write(text_block.strip())
                fh.write('\n')
        processed += 1
    print(f'Wrote intervention files for {processed} sessions to', out_dir)


In [ ]:
# Process batch
# batch_interventions_to_files(BASE_TEXT_DIR, OUT_INTERV_DIR)

Wrote intervention files for 18 sessions to ..\data\samples\interventions
